In [1]:
!git clone -b rogelio_resnet50_modeling  https://github.com/seanmcgowanx/coral-reef-classification.git

Cloning into 'coral-reef-classification'...
remote: Enumerating objects: 149, done.
remote: Counting objects: 100% (102/102), done.
remote: Compressing objects: 100% (93/93), done.
remote: Total 149 (delta 40), reused 22 (delta 5), pack-reused 47 (from 1)
Receiving objects: 100% (149/149), 65.16 MiB | 16.94 MiB/s, done.
Resolving deltas: 100% (45/45), done.


In [2]:
!pip install -q -r /content/coral-reef-classification/requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.4/14.4 MB 82.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 4.1 MB/s eta 0:00:00


# ResNet-50

In [ ]:
import sys
import os

def in_colab():
    return "COLAB_GPU" in os.environ or "google.colab" in sys.modules

if in_colab():
    # Running in Google Colab
    repo_path = "/content/coral-reef-classification"
    data_path = "/content/coral-reef-classification/data/processed"

    # Set working directory to the repo root
    os.chdir(repo_path)

else:
    # Running locally in VS Code
    repo_path = os.path.abspath(os.path.join(os.getcwd(), ".."))
    data_path = os.path.abspath(os.path.join(repo_path, "data", "processed"))

    # Add repo root to Python path
    if repo_path not in sys.path:
        sys.path.append(repo_path)

    # Set working directory to the repo root
    os.chdir(repo_path)

print("Using repo path:", repo_path)
print("Using data path:", data_path)
print("CWD:", os.getcwd())

Using repo path: /Users/rogelio/Documents/University of San Diego/AAI_501/CNNProject
Using data path: /Users/rogelio/Documents/University of San Diego/AAI_501/CNNProject/data/processed
CWD: /Users/rogelio/Documents/University of San Diego/AAI_501/CNNProject


In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from torchvision.models import resnet50, ResNet50_Weights
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

# Import custom functions
from src.s3_loader import get_image_s3
from src.save_fig import save_fig


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.5 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/opt/anaconda3/envs/coral-reef-classification/lib/python3.11/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/opt/anaconda3/envs/coral-reef-classification/lib/python3.11/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/opt/anaconda3/envs/coral-reef-classification/lib/python3.11/site-packages/ipykernel/kernelapp

### Data Augmentations

In [ ]:
# ImageNet statistics for normalization
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

# Define data augmentations and transformations
train_transform = T.Compose([
    T.RandomResizedCrop(224, scale=(0.8, 1.0)),
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.ColorJitter(
        brightness=0.3,
        contrast=0.3,
        saturation=0.3
    ),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# Define test/validation transformations
test_transform = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

### Build the CoralReefDataset Class

In [ ]:
class CoralReefDataset(Dataset):
    def __init__(self, data, transform=None):
        # Accept either a DataFrame or a CSV path
        if isinstance(data, str):
            self.df = pd.read_csv(data)
        else:
            self.df = data.reset_index(drop=True)

        # Drop region_name if it exists
        if "region_name" in self.df.columns:
            self.df = self.df.drop(columns=["region_name"])

        self.transform = transform
        self.image_ids = self.df["image_id"].tolist()
        self.label_cols = self.df.columns[1:].tolist()

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]

        # Load image from S3
        image = get_image_s3(image_id)

        # Apply transforms
        if self.transform:
            image = self.transform(image)

        # Get labels as float32 tensor
        labels = torch.tensor(
            self.df.loc[idx, self.label_cols].values.astype("float32")
        )

        return image, labels

### Create Train, Validation, Test Datasets

In [ ]:
train_csv = os.path.join(data_path, "final_labels_train.csv")
test_csv = os.path.join(data_path, "final_labels_test.csv")

# Load CSVs
df_train_full = pd.read_csv(train_csv)
df_test = pd.read_csv(test_csv)

# Extract label matrix for train / test splits
class_names = df_train_full.columns[2:]
y = df_train_full[class_names].values

msss = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=0.05,
    random_state=42
)

# Assign train / test splits
for train_idx, val_idx in msss.split(df_train_full, y):
    train_df = df_train_full.iloc[train_idx].reset_index(drop=True)
    val_df = df_train_full.iloc[val_idx].reset_index(drop=True)

# Build datasets
train_dataset = CoralReefDataset(train_df, transform=train_transform)
val_dataset = CoralReefDataset(val_df, transform=test_transform)
test_dataset = CoralReefDataset(df_test, transform=test_transform)

FileNotFoundError: [Errno 2] No such file or directory: '/Users/rogelio/Documents/University of San Diego/AAI_501/CNNProject/data/processed/final_labels_train.csv'

### Create DataLoaders

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

### Build and configure the ResNet-50 model

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load ImageNet-pretrained weights
weights = ResNet50_Weights.IMAGENET1K_V1
model = resnet50(weights=weights)

# Replace the final fully connected layer
num_features = model.fc.in_features
num_classes = len(class_names)  # should be 16
model.fc = nn.Linear(num_features, num_classes)

# Move model to device
model = model.to(device)

# Loss function for multi-label classification
criterion = nn.BCEWithLogsitsLoss()

# Optimizer
optimizer = optim.Adam(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-5
)

# Cosine Annealing LR scheduler
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=10    # cycles every 10 epochs
)

print("\nModel successfully built:")
print(model)
print(f"\nNumber of trainable parameters: "
      f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

NameError: name 'torch' is not defined

### Stage 1 Training Loop

In [ ]:
def compute_metrics(logits, targets, threshold=0.5):
    """
    Computes macro precision, recall, F1 for multi-label outputs.
    """
    probs = torch.sigmoid(logits)
    preds = (probs >= threshold).float()

    tp = (preds * targets).sum(dim=0)
    fp = (preds * (1 - targets)).sum(dim=0)
    fn = ((1 - preds) * targets).sum(dim=0)

    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)

    return (
        precision.mean().item(),
        recall.mean().item(),
        f1.mean().item()
    )


def validate(model, loader, criterion, device, threshold=0.5):
    """Run one validation epoch."""
    model.eval()

    total_loss = 0
    total_p, total_r, total_f1 = 0, 0, 0
    n_batches = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            logits = model(images)
            loss = criterion(logits, labels)

            p, r, f1 = compute_metrics(logits, labels, threshold)

            total_loss += loss.item()
            total_p += p
            total_r += r
            total_f1 += f1
            n_batches += 1

    return (
        total_loss / n_batches,
        total_p / n_batches,
        total_r / n_batches,
        total_f1 / n_batches
    )


def train_stage_1(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    scheduler,
    device,
    num_epochs=10,
    threshold=0.5,
    ckpt_path="stage1_best_model.pth"
):
    best_val_f1 = 0.0

    print("\n===== Starting Stage 1 Training =====\n")

    for epoch in range(1, num_epochs + 1):
        model.train()

        running_loss = 0.0
        running_p = 0.0
        running_r = 0.0
        running_f1 = 0.0
        n_batches = 0

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            logits = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            p, r, f1 = compute_metrics(logits.detach(), labels, threshold)

            running_loss += loss.item()
            running_p += p
            running_r += r
            running_f1 += f1
            n_batches += 1

        train_loss = running_loss / n_batches
        train_p = running_p / n_batches
        train_r = running_r / n_batches
        train_f1 = running_f1 / n_batches

        # Validation epoch
        val_loss, val_p, val_r, val_f1 = validate(
            model, val_loader, criterion, device, threshold
        )

        # LR step
        if scheduler is not None:
            scheduler.step()

        print(
            f"Epoch [{epoch}/{num_epochs}]\n"
            f"  Train Loss: {train_loss:.4f} | P: {train_p:.4f} | R: {train_r:.4f} | F1: {train_f1:.4f}\n"
            f"  Val   Loss: {val_loss:.4f} | P: {val_p:.4f} | R: {val_r:.4f} | F1: {val_f1:.4f}\n"
        )

        # Save best model based on validation F1
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "scheduler_state_dict": scheduler.state_dict() if scheduler else None,
                    "class_names": class_names.tolist(),
                },
                ckpt_path
            )
            print(f"  🔥 New best model saved! Val F1 improved to {best_val_f1:.4f}\n")

    print("===== Stage 1 Training Complete =====")
    print(f"Best Validation F1: {best_val_f1:.4f}")

    return best_val_f1

In [ ]:
best_f1 = train_stage_1(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    device=device,
    num_epochs=10,
    ckpt_path="resnet50_stage1_best.pth"
)